# Домашнее задание. Лекция 1 — с автопроверкой

7 задач по подзапросам, группировкам, CTE и оконным функциям.

### Как работать
1. Запустите первые две служебные code-ячейки: генерацию данных и автопроверку.
2. В каждой задаче пишите **только SQL внутри `sql = r"""..."""`**.
3. Запустите ячейку задачи — `check_sql(...)` сразу выставит статус и балл.
4. Проверка не требует буквального совпадения текста SQL: сравнивается результат, зерно, значения и обязательный метод там, где он указан.
5. Для почти правильных решений запускаются дополнительные тесты на других данных — это защищает от случайного hardcode.

Статусы: **полное решение / почти готово / частичное решение / неверное решение / ошибка SQL**.

> Сортировка строк сама по себе не оценивается, если она не является частью смысла задачи. Имена обязательных столбцов указаны под каждой задачей.


In [1]:
import sqlite3
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
conn = sqlite3.connect(':memory:')

channels = np.array(['organic', 'paid_ads', 'referral', 'email'])
cities = np.array(['Москва', 'Санкт-Петербург', 'Казань', 'Новосибирск', 'Екатеринбург'])
categories = np.array(['electronics', 'home', 'books', 'sport', 'beauty', 'kids'])
brands = np.array(['alpha', 'beta', 'gamma', 'delta', 'omega'])

n_users = 700
users = pd.DataFrame({
    'user_id': np.arange(1, n_users + 1),
    'signup_date': pd.to_datetime('2024-01-01') + pd.to_timedelta(rng.integers(0, 190, n_users), unit='D'),
    'signup_channel': rng.choice(channels, n_users, p=[0.38, 0.32, 0.18, 0.12]),
    'city': rng.choice(cities, n_users),
})
users['signup_date'] = users['signup_date'].dt.strftime('%Y-%m-%d')

n_products = 80
base_price = rng.integers(250, 7000, n_products)
products = pd.DataFrame({
    'product_id': np.arange(1, n_products + 1),
    'category': rng.choice(categories, n_products),
    'brand': rng.choice(brands, n_products),
    'base_price': base_price.astype(float),
})

n_orders = 2600
order_users = rng.choice(users['user_id'], n_orders)
user_signup = users.set_index('user_id').loc[order_users, 'signup_date'].to_numpy()
order_dates = pd.to_datetime(user_signup) + pd.to_timedelta(rng.integers(0, 190, n_orders), unit='D')
statuses = rng.choice(['paid', 'cancelled', 'refunded'], n_orders, p=[0.78, 0.16, 0.06])
promos = rng.choice([None, 'WELCOME10', 'SUMMER15', 'VIP20'], n_orders, p=[0.68, 0.16, 0.11, 0.05])
orders = pd.DataFrame({
    'order_id': np.arange(1, n_orders + 1),
    'user_id': order_users,
    'order_date': order_dates.strftime('%Y-%m-%d'),
    'status': statuses,
    'promo_code': promos,
})

items = []
for oid in orders['order_id']:
    k = rng.integers(1, 5)
    pids = rng.choice(products['product_id'], k, replace=False)
    for pid in pids:
        base = float(products.loc[products.product_id == pid, 'base_price'].iloc[0])
        price = round(base * rng.uniform(0.75, 1.25), 2)
        qty = int(rng.choice([1, 1, 1, 2, 3]))
        items.append((oid, int(pid), qty, price))
order_items = pd.DataFrame(items, columns=['order_id', 'product_id', 'quantity', 'price'])

events = pd.DataFrame({
    'event_id': np.arange(1, 9500),
    'user_id': rng.choice(users['user_id'], 9499),
    'event_time': (pd.to_datetime('2024-01-01') + pd.to_timedelta(rng.integers(0, 280, 9499), unit='D')).strftime('%Y-%m-%d'),
    'event_type': rng.choice(['app_open', 'product_view', 'add_to_cart', 'search'], 9499, p=[0.45, 0.32, 0.11, 0.12]),
    'product_id': rng.choice(products['product_id'], 9499),
})

for name, df in [('users', users), ('products', products), ('orders', orders), ('order_items', order_items), ('events', events)]:
    df.to_sql(name, conn, index=False, if_exists='replace')

def run(sql):
    return pd.read_sql_query(sql, conn)

def explain(sql):
    return pd.read_sql_query('EXPLAIN QUERY PLAN ' + sql, conn)

for table in ['users', 'products', 'orders', 'order_items', 'events']:
    print(table, run(f'SELECT COUNT(*) AS rows FROM {table}').iloc[0, 0])

users 700
products 80
orders 2600
order_items 6481
events 9499


In [2]:
#@title 🔒 Автопроверка — запустите один раз и не редактируйте
import re as _re
import sqlite3 as _sqlite3
import numpy as _np
import pandas as _pd

_GRADE_SPECS = {
    1: dict(
        required=['segment','users','avg_ltv','total_ltv'],
        keys=['segment'],
        values=['users','avg_ltv','total_ltv'],
        concept='',
    ),
    2: dict(
        required=['month','category','product_id','revenue','rank_in_category'],
        keys=['month','category','product_id'],
        values=['revenue','rank_in_category'],
        concept='window_rank',
    ),
    3: dict(
        required=['user_id','category'],
        keys=['user_id','category'],
        values=[],
        concept='',
    ),
    4: dict(
        required=['promo_code','paid_orders','revenue','avg_order_value'],
        keys=['promo_code'],
        values=['paid_orders','revenue','avg_order_value'],
        concept='',
    ),
    5: dict(
        required=['city','buyers','repeat_buyers','repeat_rate','overall_repeat_rate'],
        keys=['city'],
        values=['buyers','repeat_buyers','repeat_rate','overall_repeat_rate'],
        concept='',
    ),
    6: dict(
        required=['signup_channel','users','activated_users','activation_rate'],
        keys=['signup_channel'],
        values=['users','activated_users','activation_rate'],
        concept='',
    ),
    7: dict(
        required=['order_id','user_id','order_date','order_revenue','prev_avg_revenue'],
        keys=['order_id'],
        values=['user_id','order_date','order_revenue','prev_avg_revenue'],
        concept='window_avg',
    ),
}

def _strip_sql_comments(sql):
    sql = _re.sub(r'/\*.*?\*/', ' ', str(sql), flags=_re.S)
    sql = _re.sub(r'--[^\n]*', ' ', sql)
    return sql.strip()

def _validate_select(sql):
    if not isinstance(sql, str) or not sql.strip():
        return False, "Решение пока не заполнено."
    clean = _strip_sql_comments(sql).strip()
    clean = clean.rstrip(';').strip()
    if ';' in clean:
        return False, "В ячейке должен быть ровно один SELECT/WITH-запрос."
    if not _re.match(r'(?is)^(select|with)\b', clean):
        return False, "Разрешены только SELECT/WITH-запросы."
    forbidden = r'\b(insert|update|delete|drop|alter|create|attach|detach|pragma|vacuum|replace)\b'
    if _re.search(forbidden, clean, flags=_re.I):
        return False, "Запрос изменяет базу. Для этой работы разрешены только читающие SELECT/WITH-запросы."
    return True, clean

def _read_tables(c):
    names = ['users','products','orders','order_items','events']
    return {n: _pd.read_sql_query(f'SELECT * FROM {n}', c) for n in names}

def _expected(task_id, c):
    t = _read_tables(c)
    u, p, o, oi, e = t['users'], t['products'], t['orders'], t['order_items'], t['events']
    oi = oi.copy()
    oi['item_revenue'] = oi['quantity'].astype(float) * oi['price'].astype(float)

    if task_id == 1:
        paid = o[o.status.eq('paid')][['order_id','user_id']].merge(
            oi[['order_id','item_revenue']], on='order_id', how='inner'
        )
        order_rev = paid.groupby(['order_id','user_id'], as_index=False)['item_revenue'].sum()
        per_user = order_rev.groupby('user_id').agg(
            paid_orders=('order_id','nunique'), ltv=('item_revenue','sum')
        ).reset_index()
        z = u[['user_id']].merge(per_user, on='user_id', how='left').fillna({'paid_orders':0,'ltv':0.0})
        z['paid_orders'] = z['paid_orders'].astype(int)
        z['segment'] = _np.select(
            [z.paid_orders.eq(0), z.paid_orders.eq(1), z.paid_orders.between(2,3), z.paid_orders.ge(4)],
            ['no_paid_orders','one_order','2_3_orders','4_plus_orders'],
            default='other'
        )
        out = z.groupby('segment', as_index=False).agg(
            users=('user_id','size'), avg_ltv=('ltv','mean'), total_ltv=('ltv','sum')
        )
        return out

    if task_id == 2:
        paid = o[o.status.eq('paid')][['order_id','order_date']].copy()
        paid['month'] = paid['order_date'].str.slice(0,7)
        z = paid.merge(oi[['order_id','product_id','item_revenue']], on='order_id').merge(
            p[['product_id','category']], on='product_id'
        )
        g = z.groupby(['month','category','product_id'], as_index=False)['item_revenue'].sum()
        g = g.rename(columns={'item_revenue':'revenue'})
        g = g.sort_values(['month','category','revenue','product_id'], ascending=[True,True,False,True])
        g['rank_in_category'] = g.groupby(['month','category']).cumcount() + 1
        return g[g.rank_in_category.le(3)][['month','category','product_id','revenue','rank_in_category']].reset_index(drop=True)

    if task_id == 3:
        views = e[e.event_type.eq('product_view')][['user_id','product_id']].merge(
            p[['product_id','category']], on='product_id'
        )[['user_id','category']].drop_duplicates()
        bought = o[o.status.eq('paid')][['order_id','user_id']].merge(
            oi[['order_id','product_id']], on='order_id'
        ).merge(p[['product_id','category']], on='product_id')[['user_id','category']].drop_duplicates()
        out = views.merge(bought.assign(_bought=1), on=['user_id','category'], how='left')
        return out[out._bought.isna()][['user_id','category']].drop_duplicates().reset_index(drop=True)

    if task_id == 4:
        paid = o[o.status.eq('paid') & o.promo_code.notna()][['order_id','promo_code']]
        z = paid.merge(oi[['order_id','item_revenue']], on='order_id')
        order_rev = z.groupby(['order_id','promo_code'], as_index=False)['item_revenue'].sum()
        out = order_rev.groupby('promo_code', as_index=False).agg(
            paid_orders=('order_id','nunique'),
            revenue=('item_revenue','sum'),
            avg_order_value=('item_revenue','mean')
        )
        return out[out.paid_orders.ge(20)].reset_index(drop=True)

    if task_id == 5:
        cnt = o[o.status.eq('paid')].groupby('user_id').size().rename('paid_orders').reset_index()
        buyers = cnt.merge(u[['user_id','city']], on='user_id', how='left')
        buyers['is_repeat'] = buyers.paid_orders.ge(2).astype(int)
        out = buyers.groupby('city', as_index=False).agg(
            buyers=('user_id','size'), repeat_buyers=('is_repeat','sum')
        )
        out['repeat_rate'] = out['repeat_buyers'] / out['buyers']
        overall = float(buyers['is_repeat'].mean()) if len(buyers) else _np.nan
        out['overall_repeat_rate'] = overall
        return out[out.repeat_rate.gt(overall)].reset_index(drop=True)

    if task_id == 6:
        paid = o[o.status.eq('paid')][['user_id','order_date']].copy()
        paid['order_date'] = _pd.to_datetime(paid['order_date'])
        z = u[['user_id','signup_date','signup_channel']].copy()
        z['signup_date'] = _pd.to_datetime(z['signup_date'])
        j = z.merge(paid, on='user_id', how='left')
        j['days_after_signup'] = (j['order_date'] - j['signup_date']).dt.days
        j['hit'] = j['days_after_signup'].between(0,6, inclusive='both').fillna(False)
        act = j.groupby('user_id', as_index=False)['hit'].max().rename(columns={'hit':'activated'})
        z = z.merge(act, on='user_id', how='left')
        z['activated'] = z['activated'].fillna(False).astype(int)
        out = z.groupby('signup_channel', as_index=False).agg(
            users=('user_id','size'), activated_users=('activated','sum')
        )
        out['activation_rate'] = out['activated_users'] / out['users']
        return out

    if task_id == 7:
        paid = o[o.status.eq('paid')][['order_id','user_id','order_date']].merge(
            oi[['order_id','item_revenue']], on='order_id'
        )
        g = paid.groupby(['order_id','user_id','order_date'], as_index=False)['item_revenue'].sum()
        g = g.rename(columns={'item_revenue':'order_revenue'})
        g = g.sort_values(['user_id','order_date','order_id']).reset_index(drop=True)
        g['prev_avg_revenue'] = g.groupby('user_id')['order_revenue'].transform(
            lambda s: s.expanding().mean().shift(1)
        )
        out = g[g['prev_avg_revenue'].notna() & (g.order_revenue > 3 * g.prev_avg_revenue)]
        return out[['order_id','user_id','order_date','order_revenue','prev_avg_revenue']].reset_index(drop=True)

    raise ValueError("Unknown task")

def _make_validation_conn(seed):
    rng = _np.random.default_rng(seed)
    c = _sqlite3.connect(':memory:')
    channels = _np.array(['organic','paid_ads','referral','email'])
    cities = _np.array(['Москва','Санкт-Петербург','Казань','Новосибирск','Екатеринбург'])
    categories = _np.array(['electronics','home','books','sport','beauty','kids'])
    brands = _np.array(['alpha','beta','gamma','delta','omega'])

    n_users, n_products, n_orders, n_events = 160, 40, 1200, 2800
    users2 = _pd.DataFrame({
        'user_id': _np.arange(1,n_users+1),
        'signup_date': _pd.to_datetime('2024-01-01') + _pd.to_timedelta(rng.integers(0,170,n_users), unit='D'),
        'signup_channel': rng.choice(channels,n_users),
        'city': rng.choice(cities,n_users),
    })
    users2['signup_date'] = users2['signup_date'].dt.strftime('%Y-%m-%d')
    products2 = _pd.DataFrame({
        'product_id': _np.arange(1,n_products+1),
        'category': rng.choice(categories,n_products),
        'brand': rng.choice(brands,n_products),
        'base_price': rng.integers(150,9000,n_products).astype(float),
    })
    order_users = rng.choice(users2.user_id,n_orders)
    signup_map = users2.set_index('user_id')['signup_date']
    offsets = rng.integers(0,190,n_orders)
    order_dates = _pd.to_datetime(signup_map.loc[order_users].to_numpy()) + _pd.to_timedelta(offsets,unit='D')
    orders2 = _pd.DataFrame({
        'order_id': _np.arange(1,n_orders+1),
        'user_id': order_users,
        'order_date': order_dates.strftime('%Y-%m-%d'),
        'status': rng.choice(['paid','cancelled','refunded'],n_orders,p=[.77,.16,.07]),
        'promo_code': rng.choice([None,'WELCOME10','SUMMER15','VIP20'],n_orders,p=[.63,.17,.13,.07]),
    })
    items=[]
    base_map=products2.set_index('product_id')['base_price'].to_dict()
    for oid in orders2.order_id:
        k=int(rng.integers(1,5))
        pids=rng.choice(products2.product_id,k,replace=False)
        for pid in pids:
            items.append((int(oid),int(pid),int(rng.choice([1,1,2,3])),round(float(base_map[int(pid)])*rng.uniform(.65,1.35),2)))
    items2=_pd.DataFrame(items,columns=['order_id','product_id','quantity','price'])

    # Гарантированно создаем хотя бы один "аномально крупный" поздний paid-заказ.
    paid_counts=orders2[orders2.status.eq('paid')].groupby('user_id').size()
    candidates=paid_counts[paid_counts.ge(2)].index
    if len(candidates):
        uid=int(candidates[0])
        oo=orders2[(orders2.user_id.eq(uid)) & orders2.status.eq('paid')].sort_values(['order_date','order_id'])
        target=int(oo.iloc[-1].order_id)
        items2.loc[items2.order_id.eq(target),'price'] *= 25

    events2=_pd.DataFrame({
        'event_id': _np.arange(1,n_events+1),
        'user_id': rng.choice(users2.user_id,n_events),
        'event_time': (_pd.to_datetime('2024-01-01') + _pd.to_timedelta(rng.integers(0,270,n_events),unit='D')).strftime('%Y-%m-%d'),
        'event_type': rng.choice(['app_open','product_view','add_to_cart','search'],n_events,p=[.43,.34,.11,.12]),
        'product_id': rng.choice(products2.product_id,n_events),
    })
    for name,df in [('users',users2),('products',products2),('orders',orders2),('order_items',items2),('events',events2)]:
        df.to_sql(name,c,index=False,if_exists='replace')
    return c

def _lower_columns(df):
    z=df.copy()
    z.columns=[str(x).strip().lower() for x in z.columns]
    return z

def _key_set(df, cols):
    if not cols:
        return set()
    def norm(v):
        if _pd.isna(v): return '<NULL>'
        if isinstance(v, (float, _np.floating)) and _np.isfinite(v):
            return f'{float(v):.8g}'
        return str(v)
    return {tuple(norm(v) for v in row) for row in df[cols].itertuples(index=False, name=None)}

def _value_accuracy(got, exp, keys, vals):
    if not vals:
        return 1.0
    if not keys:
        return 0.0
    g=got[keys+vals].drop_duplicates(subset=keys, keep='first')
    e=exp[keys+vals].drop_duplicates(subset=keys, keep='first')
    m=e.merge(g,on=keys,how='left',suffixes=('_exp','_got'),indicator=True)
    scores=[]
    for col in vals:
        a=m[f'{col}_exp']
        b=m[f'{col}_got']
        if _pd.api.types.is_numeric_dtype(a):
            aa=_pd.to_numeric(a,errors='coerce').to_numpy(dtype=float)
            bb=_pd.to_numeric(b,errors='coerce').to_numpy(dtype=float)
            ok=_np.isclose(aa,bb,rtol=1e-5,atol=1e-4,equal_nan=True)
        else:
            ok=(a.fillna('<NULL>').astype(str).to_numpy()==b.fillna('<NULL>').astype(str).to_numpy())
        ok = ok & m['_merge'].eq('both').to_numpy()
        scores.append(float(ok.mean()) if len(ok) else 1.0)
    return float(_np.mean(scores)) if scores else 1.0

def _concept_score(task_id, sql):
    s=_strip_sql_comments(sql).lower()
    if task_id==2:
        has_over=bool(_re.search(r'\bover\s*\(',s))
        has_rank=bool(_re.search(r'\b(row_number|rank|dense_rank)\s*\(',s))
        return (10 if has_over and has_rank else 0,
                "" if has_over and has_rank else "По условию нужна оконная функция ранжирования (`ROW_NUMBER`/`RANK`/`DENSE_RANK`).")
    if task_id==7:
        has_avg=bool(_re.search(r'\bavg\s*\([^)]*\)\s*over\s*\(',s,_re.S))
        return (10 if has_avg else 0,
                "" if has_avg else "По условию среднее предыдущих заказов нужно посчитать оконной функцией `AVG(...) OVER (...)`.")
    return 10, ""

def _exact_required(got, exp, spec):
    got=_lower_columns(got); exp=_lower_columns(exp)
    req=spec['required']
    if any(c not in got.columns for c in req): return False
    got=got[req].copy(); exp=exp[req].copy()
    keys=spec['keys']; vals=spec['values']
    if len(got)!=len(exp): return False
    if len(_key_set(got,keys)) != len(got): return False
    if _key_set(got,keys) != _key_set(exp,keys): return False
    return _value_accuracy(got,exp,keys,vals) >= 0.999999

def _status(score):
    if score >= 99.5: return "✅ ПОЛНОЕ РЕШЕНИЕ"
    if score >= 80: return "🟡 ПОЧТИ ГОТОВО"
    if score >= 45: return "🟠 ЧАСТИЧНОЕ РЕШЕНИЕ"
    return "❌ НЕВЕРНОЕ РЕШЕНИЕ"

def check_sql(task_id, sql, *, show_preview=True):
    ok, clean = _validate_select(sql)
    print("─"*72)
    print(f"Задача {task_id}")
    if not ok:
        if clean == "Решение пока не заполнено.":
            print("📝 РЕШЕНИЕ ПОКА НЕ ЗАПОЛНЕНО")
        else:
            print("⚠️ SQL НЕ ПРИНЯТ")
            print(clean)
        return {'score':0,'status':'not_submitted'}

    spec=_GRADE_SPECS[task_id]
    try:
        got=_pd.read_sql_query(clean,conn)
    except Exception as exc:
        print("⚠️ ОШИБКА SQL")
        print(f"{type(exc).__name__}: {exc}")
        return {'score':0,'status':'sql_error'}

    exp=_expected(task_id,conn)
    g=_lower_columns(got); e=_lower_columns(exp)
    req=spec['required']
    present=[c for c in req if c in g.columns]
    missing=[c for c in req if c not in g.columns]
    schema_ratio=len(present)/len(req)
    schema_points=15*schema_ratio

    if missing:
        concept_points,concept_msg=_concept_score(task_id,clean)
        score=round(schema_points + concept_points,1)
        print(_status(score),f"— {score:.1f}/100")
        print("Не хватает обязательных столбцов:", ", ".join(missing))
        print("Ожидаются:", ", ".join(req))
        if concept_msg: print("Требование метода:",concept_msg)
        if show_preview and len(got):
            print("\nВаш результат (первые 5 строк):")
            display(got.head())
        return {'score':score,'status':_status(score),'missing_columns':missing}

    g2=g[req].copy(); e2=e[req].copy()
    keys=spec['keys']; vals=spec['values']
    gs=_key_set(g2,keys); es=_key_set(e2,keys)
    inter=len(gs & es)
    precision=inter/len(gs) if gs else (1.0 if not es else 0.0)
    recall=inter/len(es) if es else (1.0 if not gs else 0.0)
    key_f1=(2*precision*recall/(precision+recall)) if (precision+recall) else 0.0
    duplicate_penalty = 1.0 if len(gs)==len(g2) else max(0.5, len(gs)/max(len(g2),1))
    value_acc=_value_accuracy(g2,e2,keys,vals)
    concept_points,concept_msg=_concept_score(task_id,clean)

    score = 15 + 35*key_f1*duplicate_penalty + 40*value_acc*key_f1 + concept_points

    # Дополнительные датасеты запускаем только для уже почти правильных решений.
    robustness_passed=0
    robustness_total=0
    if score >= 90:
        for seed in (31415,27182):
            c2=_make_validation_conn(seed)
            try:
                gtest=_pd.read_sql_query(clean,c2)
                etest=_expected(task_id,c2)
                passed=_exact_required(gtest,etest,spec)
            except Exception:
                passed=False
            finally:
                c2.close()
            robustness_total += 1
            robustness_passed += int(passed)
        if robustness_passed < robustness_total:
            score=min(score, 85 if robustness_passed==1 else 75)

    score=round(min(100,max(0,score)),1)
    exact_main=_exact_required(g2,e2,spec)
    status=_status(score)
    print(status,f"— {score:.1f}/100")
    print(f"Схема: {'OK' if not missing else 'есть проблемы'} | "
          f"ключи/строки: precision={precision:.2%}, recall={recall:.2%} | "
          f"значения: {value_acc:.2%}")
    if len(gs)!=len(g2):
        print("Есть дубликаты по ключевым столбцам — проверьте зерно результата.")
    if concept_msg:
        print("Требование метода:",concept_msg)
    if robustness_total:
        print(f"Дополнительные тесты на других данных: {robustness_passed}/{robustness_total}")
        if robustness_passed < robustness_total:
            print("Результат похож на правильный на основном наборе, но решение не обобщается на новые данные.")
    if not exact_main and not missing:
        missing_keys=list(es-gs)[:3]
        extra_keys=list(gs-es)[:3]
        if missing_keys: print("Примеры пропущенных ключей:",missing_keys)
        if extra_keys: print("Примеры лишних ключей:",extra_keys)
    if show_preview:
        print("\nВаш результат (первые 5 строк):")
        display(got.head())
    return {'score':score,'status':status,'robustness':(robustness_passed,robustness_total)}

print("Автопроверка готова ✅")


Автопроверка готова ✅


## Задача 1. LTV-сегменты

Для каждого пользователя посчитайте оплаченный LTV и количество оплаченных заказов. Разбейте пользователей на сегменты: `no_paid_orders`, `one_order`, `2_3_orders`, `4_plus_orders`. Выведите размер сегмента, средний LTV и суммарный LTV.

**Важно:** пользователи без оплаченных заказов тоже должны попасть в расчёт, их LTV = 0.

**Обязательные столбцы результата:**  
`segment`, `users`, `avg_ltv`, `total_ltv`


In [3]:
sql = r"""
with orders_stat as (
    select 
        o.order_id, o.user_id,
        sum(oi.quantity * oi.price) as revenue,
        o.status
    from 
        orders o
        left join 
        order_items oi
            on o.order_id = oi.order_id
    where status = 'paid'
    group by 
        o.order_id
),
user_info as (
    select 
        u.user_id,
        (case 
            when count(os.status) = 0 then 'no_paid_orders'
            when count(os.status) = 1 then 'one_order'
            when (count(os.status) > 1 and count(os.status) < 4) then '2_3_orders'
            else '4_plus_orders' 
        end) as segment,
        sum(revenue) as user_ltv
    from 
        users u
        left join 
        orders_stat os
            on u.user_id = os.user_id
    group by 
        u.user_id
)
select 
    segment, 
    count(segment) as users, 
    coalesce(sum(user_ltv) / count(segment), 0) as avg_ltv, 
    coalesce(sum(user_ltv),0) as total_ltv  
from 
    user_info
group by 
    segment
"""

#run(sql)
check_sql(1, sql)


────────────────────────────────────────────────────────────────────────
Задача 1
✅ ПОЛНОЕ РЕШЕНИЕ — 100.0/100
Схема: OK | ключи/строки: precision=100.00%, recall=100.00% | значения: 100.00%
Дополнительные тесты на других данных: 2/2

Ваш результат (первые 5 строк):


,segment,users,avg_ltv,total_ltv
0,2_3_orders,315,38020.888857,11976579.99
1,4_plus_orders,237,75458.640717,17883697.85
2,no_paid_orders,38,0.000000,0.00
3,one_order,110,15125.178545,1663769.64


{'score': 100, 'status': '✅ ПОЛНОЕ РЕШЕНИЕ', 'robustness': (2, 2)}

## Задача 2. Top-3 товара внутри категории по месячной выручке

Для каждого месяца и категории найдите 3 товара с максимальной оплаченной выручкой. Используйте **оконную функцию** для ранжирования.

При равной выручке используйте `product_id` по возрастанию как tie-breaker.

**Обязательные столбцы результата:**  
`month`, `category`, `product_id`, `revenue`, `rank_in_category`


In [4]:
sql = r"""
with monthly_category as (
    select 
        substr(o.order_date, 1, 7) as month,
        p.category,
        oi.product_id,
        sum(oi.quantity * oi.price) as revenue
    from
        orders o
        left join
        order_items oi
            on o.order_id = oi.order_id
        left join
        products p
            on oi.product_id = p.product_id
    where 
        o.status = 'paid'
    group by
        substr(o.order_date, 1, 7), p.category, oi.product_id
)
select 
    * 
from( 
    select 
        month,
        category, 
        product_id,
        revenue,
        rank() over (partition by month, category order by revenue desc) as rank_in_category
    from 
        monthly_category
) sub
where rank_in_category <= 3
"""
#run(sql)
check_sql(2, sql)


────────────────────────────────────────────────────────────────────────
Задача 2
✅ ПОЛНОЕ РЕШЕНИЕ — 100.0/100
Схема: OK | ключи/строки: precision=100.00%, recall=100.00% | значения: 100.00%
Дополнительные тесты на других данных: 2/2

Ваш результат (первые 5 строк):


,month,category,product_id,revenue,rank_in_category
0,2024-01,beauty,55,18284.62,1
1,2024-01,beauty,57,11347.22,2
2,2024-01,beauty,22,7057.18,3
3,2024-01,books,78,25463.75,1
4,2024-01,books,35,16049.00,2


{'score': 100, 'status': '✅ ПОЛНОЕ РЕШЕНИЕ', 'robustness': (2, 2)}

## Задача 3. Просматривали категорию, но не покупали ее

Найдите пары `user_id + category`, где пользователь смотрел товары категории (`events.event_type = 'product_view'`), но ни разу не покупал **оплаченный** товар этой категории.

**Обязательные столбцы результата:**  
`user_id`, `category`


In [5]:
sql = r"""
select 
    *
from
    events
   
"""
run(sql)

,event_id,user_id,event_time,event_type,product_id
0,1,370,2024-02-21,product_view,29
1,2,228,2024-03-24,product_view,74
2,3,390,2024-07-12,app_open,49
3,4,241,2024-05-29,add_to_cart,63
4,5,32,2024-02-26,search,69
...,...,...,...,...,...
9494,9495,429,2024-02-16,app_open,26
9495,9496,589,2024-08-31,app_open,40
9496,9497,80,2024-03-31,app_open,54
9497,9498,12,2024-07-13,add_to_cart,26


In [6]:
sql = r"""
select 
    user_id, category
from 
    events e
    left join
    products p
        on p.product_id = e.product_id
where event_type = 'product_view'
except
select 
    user_id, category
from 
    orders o
    left join
    order_items oi
        on o.order_id = oi.order_id    
    left join
        products p
            on p.product_id = oi.product_id
where status = 'paid'
"""

#run(sql)
check_sql(3, sql)


────────────────────────────────────────────────────────────────────────
Задача 3
✅ ПОЛНОЕ РЕШЕНИЕ — 100.0/100
Схема: OK | ключи/строки: precision=100.00%, recall=100.00% | значения: 100.00%
Дополнительные тесты на других данных: 2/2

Ваш результат (первые 5 строк):


,user_id,category
0,1,home
1,2,books
2,3,electronics
3,5,beauty
4,6,beauty


{'score': 100, 'status': '✅ ПОЛНОЕ РЕШЕНИЕ', 'robustness': (2, 2)}

## Задача 4. Промокоды

Для каждого **не-NULL** промокода посчитайте количество оплаченных заказов, выручку и средний чек. Оставьте только промокоды с 20+ оплаченными заказами.

Средний чек считается на уровне заказа, а не строки `order_items`.

**Обязательные столбцы результата:**  
`promo_code`, `paid_orders`, `revenue`, `avg_order_value`


In [7]:
sql = r"""
select 
    promo_code, 
    count(distinct o.order_id) as paid_orders, 
    sum(oi.quantity * oi.price) as revenue, 
    (sum(oi.quantity * oi.price) / count(distinct o.order_id)) as avg_order_value
from 
    orders o
    left join
    order_items oi
        on o.order_id = oi.order_id
where 
    status = 'paid' and promo_code IS NOT NULL
group by 
    promo_code
having count(distinct o.order_id) >= 20
"""
#run(sql)
check_sql(4, sql)


────────────────────────────────────────────────────────────────────────
Задача 4
✅ ПОЛНОЕ РЕШЕНИЕ — 100.0/100
Схема: OK | ключи/строки: precision=100.00%, recall=100.00% | значения: 100.00%
Дополнительные тесты на других данных: 2/2

Ваш результат (первые 5 строк):


,promo_code,paid_orders,revenue,avg_order_value
0,SUMMER15,223,3446366.39,15454.557803
1,VIP20,116,1732825.06,14938.147069
2,WELCOME10,331,4820101.93,14562.241480


{'score': 100, 'status': '✅ ПОЛНОЕ РЕШЕНИЕ', 'robustness': (2, 2)}

## Задача 5. Города с repeat-rate выше среднего

Repeat-rate = доля покупателей города, у которых 2+ оплаченных заказа. Покупатель — пользователь, у которого есть хотя бы один оплаченный заказ.

Найдите города, где repeat-rate выше общего repeat-rate по всем покупателям.

**Обязательные столбцы результата:**  
`city`, `buyers`, `repeat_buyers`, `repeat_rate`, `overall_repeat_rate`


In [8]:
sql = r"""
with users_info as (
    select 
        city, 
        u.user_id,
        count(u.user_id) as paid_orders 
    from 
        users u
        left join 
        orders o 
            on o.user_id = u.user_id
    where 
        status = 'paid'
    group by 
        city, u.user_id
    order by 
        u.user_id
),
city_buyers as (
    select 
        city, 
        count( user_id) as buyers,
        sum(
            case when paid_orders > 1 then 1 else 0 end
        ) as repeat_buyers
    from 
        users_info
    group by 
        city
),
rates as (
    select 
        *, (repeat_buyers * 1.0 /buyers ) as repeat_rate, sum(repeat_buyers) over() * 1.0 /sum(buyers) over() as overall_repeat_rate
    from 
        city_buyers
)
select 
    * 
from 
    rates
where 
    repeat_rate > overall_repeat_rate;
"""

#run(sql)
check_sql(5, sql)


────────────────────────────────────────────────────────────────────────
Задача 5
✅ ПОЛНОЕ РЕШЕНИЕ — 100.0/100
Схема: OK | ключи/строки: precision=100.00%, recall=100.00% | значения: 100.00%
Дополнительные тесты на других данных: 2/2

Ваш результат (первые 5 строк):


,city,buyers,repeat_buyers,repeat_rate,overall_repeat_rate
0,Москва,134,112,0.835821,0.833837
1,Новосибирск,113,100,0.884956,0.833837
2,Санкт-Петербург,134,113,0.843284,0.833837


{'score': 100, 'status': '✅ ПОЛНОЕ РЕШЕНИЕ', 'robustness': (2, 2)}

## Задача 6. Активация в первые 7 дней

Пользователь активирован, если сделал оплаченный заказ в первые 7 дней после регистрации.

Для проверки используем интервал **[signup_date, signup_date + 7 дней)**, то есть дни 0–6 включительно.

Посчитайте activation rate по каналам привлечения.

**Обязательные столбцы результата:**  
`signup_channel`, `users`, `activated_users`, `activation_rate`


In [9]:
sql = r"""

select 
    signup_channel,
    count(distinct u.user_id) as users,
    count(
        distinct case 
            when o.status = 'paid' and order_date <= date(signup_date, '+6 days') then u.user_id 
        end
    ) as activated_users,
    count(
        distinct case 
            when o.status = 'paid' and order_date <= date(signup_date, '+6 days') then u.user_id 
        end
    ) * 1.0 / count(distinct u.user_id) as activation_rate
from 
    users u
    left join 
    orders o
        on u.user_id = o.user_id
group by 
    signup_channel

"""
#run(sql)
check_sql(6, sql)

────────────────────────────────────────────────────────────────────────
Задача 6
✅ ПОЛНОЕ РЕШЕНИЕ — 100.0/100
Схема: OK | ключи/строки: precision=100.00%, recall=100.00% | значения: 100.00%
Дополнительные тесты на других данных: 2/2

Ваш результат (первые 5 строк):


,signup_channel,users,activated_users,activation_rate
0,email,88,14,0.159091
1,organic,264,24,0.090909
2,paid_ads,221,21,0.095023
3,referral,127,14,0.110236


{'score': 100, 'status': '✅ ПОЛНОЕ РЕШЕНИЕ', 'robustness': (2, 2)}

## Задача 7. Аномально крупный заказ пользователя

Для каждого оплаченного заказа посчитайте выручку заказа. Найдите заказы, где выручка больше чем в 3 раза превышает среднюю выручку **предыдущих** оплаченных заказов этого пользователя. Первый заказ пользователя исключите.

Для порядка заказов используйте `ORDER BY order_date, order_id`. Среднее предыдущих заказов нужно посчитать **оконной функцией `AVG(...) OVER (...)`**.

**Обязательные столбцы результата:**  
`order_id`, `user_id`, `order_date`, `order_revenue`, `prev_avg_revenue`


In [ ]:
sql = r"""
with order_info as (
    select 
        o.order_id, 
        user_id, 
        order_date,
        sum(quantity*price) as order_revenue
    from 
        orders o
        left join 
        order_items oi 
        on o.order_id = oi.order_id
    where 
        status = 'paid'
    group by 
        o.order_id, 
        user_id, 
        order_date
),
order_prev as (
    select 
        *,
        avg(order_revenue) over (partition by user_id ORDER BY order_date, order_id ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) as prev_avg_revenue
    from 
        order_info
    order by user_id, order_date desc
)
select 
    * 
from 
    order_prev
where order_revenue > prev_avg_revenue * 3
"""

#run(sql)
check_sql(7, sql)


────────────────────────────────────────────────────────────────────────
Задача 7
✅ ПОЛНОЕ РЕШЕНИЕ — 100.0/100
Схема: OK | ключи/строки: precision=100.00%, recall=100.00% | значения: 100.00%
Дополнительные тесты на других данных: 2/2

Ваш результат (первые 5 строк):


,order_id,user_id,order_date,order_revenue,prev_avg_revenue
0,1649,6,2024-09-03,14030.07,3934.080
1,1891,11,2024-06-29,21630.36,6836.450
2,957,18,2024-07-06,30668.76,9033.910
3,1120,22,2024-09-08,12949.88,3026.115
4,1711,29,2024-09-08,16577.99,3419.060


{'score': 100, 'status': '✅ ПОЛНОЕ РЕШЕНИЕ', 'robustness': (2, 2)}